# scRareBench | High-Level scDML | Dataset 4 — COVID-19 autoimmunity PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This notebook runs **scDML** on scRareBench Dataset 4 (`covid19_autoimmunity_pbmc`) using:

`load_dataset(4) → MethodSpec(scDML) → benchmark_method() → scIB + rare benchmark → HTML / multi-seed bundle`

Pinned scRareBench commit:

`cfdb720f5488b9ceef9a225b35fc7558bcb0251c`

## Dataset 4 contract

- 97,499 cells × 21,858 features
- benchmark label: `cell_type` — evaluation only, never supplied to scDML
- integration/evaluation batch: `donor_id`
- canonical counts: `layers["counts"]`
- 22 donors
- 26 biological label categories
- registered rare-scenario metadata: package-owned
- scIB-reference HVG mode: `evaluation_batch`

## scDML policy

The method settings are kept consistent with the Dataset 3 scDML workflow:

- Leiden initialization, resolution 3.0
- native scDML preprocessing with 2,000 HVGs
- normalize total 1e4, log1p
- batch-wise scaling, max value 10
- PCA 50
- fully unsupervised (`celltype_key=None`)
- K_in=5, K_bw=10
- cosine neighbor metrics
- merge rule `rule2`
- 50 epochs
- hard triplets, Euclidean metric, margin 0.2
- output dimension 32
- hidden layer [256]

scDML's legacy preprocessing may reorder rows. An explicit original-barcode guard
is carried through preprocessing, and the final latent is restored to the exact
original scRareBench row order before `MethodOutput`.

Recommended runtime: **GPU + High-RAM Colab**.
For a first smoke run, temporarily set `METHOD_SEEDS = [42]`.

## 1. Install pinned scRareBench and scDML runtimes

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCDML_REVISION = "d762dd75557c0dbb2d3039174d69a18195842b93"

SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)
SCDML_GITHUB = (
    "git+https://github.com/eleozzr/"
    f"scDML.git@{SCDML_REVISION}"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_RUNTIME_DEPENDENCIES = (
    "tables>=3.6.1",
    "numpy-groupies>=0.9.14",
    "numexpr>=2.7.1",
    "networkx>=2.5",
    "seaborn>=0.13",
    "torch>=2.0",
    "pytorch-metric-learning==2.9.0",
    "annoy @ git+https://github.com/spotify/annoy.git@379f744667aba6b40ba3db8a07678df173a88f74",
    "hnswlib @ git+https://github.com/nmslib/hnswlib.git@ca426729609b3221563047ceb269cc1213e0d376",
)

from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_RUNTIME_DEPENDENCIES,
    extra_imports=(
        "tables",
        "numpy_groupies",
        "numexpr",
        "networkx",
        "seaborn",
        "torch",
        "pytorch_metric_learning",
        "annoy",
        "hnswlib",
    ),
    quiet=False,
)
print_install_report(install_report)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCDML_GITHUB,
])

subprocess.check_call([
    sys.executable,
    "-c",
    (
        "import scDML, torch; "
        "from scDML import scDMLModel; "
        "print('scDML', getattr(scDML, '__version__', 'unknown')); "
        "print('torch', torch.__version__)"
    ),
])


## 2. Load Dataset 4 through scRareBench

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset, scenario_table_from_adata

adata = load_dataset(4)
info = dataset_info(adata)

expected = {
    "dataset_key": "covid19_autoimmunity_pbmc",
    "label_key": "cell_type",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "evaluation_batch",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 4 contract for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (97_499, 21_858):
    raise RuntimeError(
        f"Unexpected Dataset 4 shape: {adata.shape}; expected (97499, 21858)."
    )
if "counts" not in adata.layers:
    raise RuntimeError("Dataset 4 canonical raw-count layer 'counts' is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Dataset 4 observation names are not unique.")
if not adata.var_names.is_unique:
    raise RuntimeError("Dataset 4 feature names are not unique.")
if adata.obs["donor_id"].astype(str).nunique() != 22:
    raise RuntimeError("Unexpected Dataset 4 donor cardinality; expected 22.")
if adata.obs["cell_type"].astype(str).nunique() != 26:
    raise RuntimeError("Unexpected Dataset 4 label cardinality; expected 26.")

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or scenario_table.empty:
    raise RuntimeError("Dataset 4 registered scenario metadata is missing.")
if len(scenario_table) != 12:
    raise RuntimeError(
        f"Unexpected Dataset 4 scenario-table size: {len(scenario_table)}; "
        "expected 12."
    )

print("scRareBench:", scrarebench.__version__)
print(adata)
print("Donors:", adata.obs["donor_id"].astype(str).nunique())
print("Cell-type categories:", adata.obs["cell_type"].astype(str).nunique())
print("Registered six-state rows:", len(scenario_table))


## 3. Reproducibility and scDML configuration

The method-side preprocessing below is scDML-owned and remains separate from
scRareBench's scIB reference preprocessing.

- scDML method preprocessing: native 2,000-HVG path
- scIB reference: package-controlled 4,000 HVGs using Dataset 4
  `scib_hvg_batch_mode="evaluation_batch"`


In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "cluster_method": "leiden",
    "resolution": 3.0,
    "n_high_var": 2000,
    "normalize_samples": True,
    "target_sum": 1e4,
    "log_normalize": True,
    "normalize_features": True,
    "pca_dim": 50,
    "scale_value": 10.0,

    "K_in": 5,
    "K_bw": 10,
    "K_in_metric": "cosine",
    "K_bw_metric": "cosine",
    "merge_rule": "rule2",

    "num_epochs": 50,
    "projection": False,
    "early_stop": False,
    "batch_size": 64,
    "metric": "euclidean",
    "margin": 0.2,
    "triplet_type": "hard",
    "device": None,

    "out_dim": 32,
    "emb_dim": [256],
    "save_model": False,
}

OUTPUT_DIR = "/content/scrarebench_scdml_dataset4_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_scdml_dataset4_method_runs"


## 4. User-owned scDML adapter

scDML can internally reorder observations during its legacy preprocessing.
The `_scrarebench_original_barcode` guard preserves original identity and the
final latent is reordered back before returning `MethodOutput`.

The benchmark biological label is never given to scDML.


In [ ]:
from pathlib import Path
import gc
import random
import shutil

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

from scDML import scDMLModel
from scrarebench import MethodOutput, dataset_info

_ORIGINAL_BARCODE_COL = "_scrarebench_original_barcode"


def _light_raw_count_guard(matrix, *, seed, max_values=100_000):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).reshape(-1)
    values = np.asarray(values)

    if values.size > max_values:
        rng = np.random.default_rng(int(seed))
        values = values[
            rng.choice(values.size, size=int(max_values), replace=False)
        ]

    values = values.astype(float, copy=False)
    if values.size == 0:
        return
    if not np.isfinite(values).all():
        raise ValueError("Canonical count matrix contains NaN or infinite values.")
    if float(values.min()) < 0:
        raise ValueError("Canonical count matrix contains negative values.")
    if not np.allclose(values, np.rint(values), atol=1e-6, rtol=0.0):
        raise ValueError("Canonical count matrix does not look integer-like.")


def run_scdml(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "covid19_autoimmunity_pbmc":
        raise RuntimeError(
            "This runner is configured only for scRareBench Dataset 4."
        )
    if batch_key != "donor_id":
        raise RuntimeError(
            f"Unexpected Dataset 4 batch key {batch_key!r}; expected 'donor_id'."
        )
    if method_info.get("scib_hvg_batch_mode") != "evaluation_batch":
        raise RuntimeError(
            "Dataset 4 is expected to use evaluation-batch-aware scIB HVGs."
        )
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(
            f"Canonical raw-count layer {count_layer!r} is unavailable."
        )

    original_order = method_adata.obs_names.astype(str).to_numpy(copy=True)
    if len(np.unique(original_order)) != len(original_order):
        raise RuntimeError("Benchmark cell identifiers must be unique.")

    method_adata.obs[_ORIGINAL_BARCODE_COL] = original_order

    counts = method_adata.layers[count_layer]
    _light_raw_count_guard(counts, seed=seed)

    # scDML.preprocess() consumes X; this is a per-seed isolated copy.
    method_adata.X = counts.copy()

    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = scDMLModel(
        verbose=True,
        save_dir=str(seed_dir) + "/",
    )

    adata_scdml = model.preprocess(
        method_adata,
        cluster_method=str(config["cluster_method"]),
        resolution=float(config["resolution"]),
        batch_key=batch_key,
        n_high_var=int(config["n_high_var"]),
        normalize_samples=bool(config["normalize_samples"]),
        target_sum=float(config["target_sum"]),
        log_normalize=bool(config["log_normalize"]),
        normalize_features=bool(config["normalize_features"]),
        pca_dim=int(config["pca_dim"]),
        scale_value=float(config["scale_value"]),
        mode="unsupervised",
    )

    if _ORIGINAL_BARCODE_COL not in adata_scdml.obs.columns:
        raise RuntimeError(
            "Original barcode guard was lost during scDML preprocessing."
        )
    if "init_cluster" not in adata_scdml.obs.columns:
        raise RuntimeError("scDML preprocessing did not create init_cluster.")

    num_init_clusters = int(adata_scdml.obs["init_cluster"].nunique())
    if num_init_clusters < 2:
        raise RuntimeError(
            f"scDML produced too few initial clusters: {num_init_clusters}"
        )

    ncluster_candidates = list(range(2, num_init_clusters + 1))

    print(
        f"Seed {seed}: cells={adata_scdml.n_obs:,}, "
        f"features={adata_scdml.n_vars:,}, "
        f"batches={adata_scdml.obs[batch_key].nunique()}, "
        f"initial Leiden clusters={num_init_clusters}"
    )

    model.integrate(
        adata_scdml,
        batch_key=batch_key,
        ncluster_list=ncluster_candidates,
        expect_num_cluster=None,
        K_in=int(config["K_in"]),
        K_bw=int(config["K_bw"]),
        K_in_metric=str(config["K_in_metric"]),
        K_bw_metric=str(config["K_bw_metric"]),
        merge_rule=str(config["merge_rule"]),
        num_epochs=int(config["num_epochs"]),
        projection=bool(config["projection"]),
        early_stop=bool(config["early_stop"]),
        batch_size=int(config["batch_size"]),
        metric=str(config["metric"]),
        margin=float(config["margin"]),
        triplet_type=str(config["triplet_type"]),
        device=config["device"],
        seed=int(seed),
        out_dim=int(config["out_dim"]),
        emb_dim=list(config["emb_dim"]),
        save_model=bool(config["save_model"]),
        celltype_key=None,
        mode="unsupervised",
    )

    if "X_emb" not in adata_scdml.obsm:
        raise RuntimeError("scDML did not create adata.obsm['X_emb'].")

    latent_native = np.asarray(
        adata_scdml.obsm["X_emb"],
        dtype=np.float32,
    )
    native_barcodes = (
        adata_scdml.obs[_ORIGINAL_BARCODE_COL]
        .astype(str)
        .to_numpy()
    )

    if latent_native.ndim != 2:
        raise RuntimeError(
            f"scDML latent must be 2D; observed shape={latent_native.shape}."
        )
    if latent_native.shape[0] != len(native_barcodes):
        raise RuntimeError(
            "scDML latent rows do not match post-preprocessing cells."
        )
    if latent_native.shape[1] != int(config["out_dim"]):
        raise RuntimeError(
            f"Unexpected scDML latent dimension {latent_native.shape[1]}; "
            f"expected {int(config['out_dim'])}."
        )
    if not np.isfinite(latent_native).all():
        raise ValueError("scDML latent contains NaN or infinite values.")
    if len(np.unique(native_barcodes)) != len(native_barcodes):
        raise RuntimeError(
            "Cell identifiers are no longer unique after scDML preprocessing."
        )
    if set(native_barcodes) != set(original_order):
        raise RuntimeError(
            "scDML preprocessing changed the benchmark cell set."
        )

    position = pd.Series(
        np.arange(len(native_barcodes)),
        index=native_barcodes,
    )
    reorder = position.loc[original_order].to_numpy()
    latent = latent_native[reorder]

    if not np.array_equal(native_barcodes[reorder], original_order):
        raise RuntimeError(
            "Failed to restore scDML latent to original benchmark order."
        )
    if not np.isfinite(latent).all():
        raise ValueError(
            "Aligned scDML latent contains NaN or infinite values."
        )

    provenance = {}
    log_path = seed_dir / "log.txt"
    if log_path.exists() and log_path.stat().st_size > 0:
        provenance["scdml_log"] = log_path

    print(f"Seed {seed}: aligned scDML latent shape={latent.shape}")

    del model, adata_scdml, latent_native, position, reorder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=original_order,
        representation_key="X_scdml",
        provenance_files=provenance,
    )


## 5. Run the high-level multi-seed benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scDML",
    runner=run_scdml,
    config=METHOD_CONFIG,
    dependencies=METHOD_RUNTIME_DEPENDENCIES + (SCDML_GITHUB,),
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        # Dataset 4 metadata supplies:
        # scib_hvg_batch_mode="evaluation_batch".
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    # scDML mutates preprocessing state; isolate each seed.
    copy_input_per_seed=True,
)


## 6. Inspect the multi-seed result

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Method:", result.method)
print("Completed seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  alignment:", run.alignment)
    print("  output:", run.output_dir)
    print("  interactive report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)

    if run.representation_key != "X_scdml":
        raise RuntimeError(
            f"Seed {seed} returned an unexpected representation key."
        )
    if run.alignment.get("exact_order_match") is not True:
        raise RuntimeError(
            f"Seed {seed} did not preserve exact barcode order."
        )


## 7. Download final multi-seed archive

In [ ]:
from pathlib import Path
import shutil
import zipfile
import os

# پیدا کردن ZIP نهایی
matches = list(
    Path("/content").rglob(
        "scDML_covid19_autoimmunity_pbmc_scRareBench_v0_10_6_multiseed_results.zip"
    )
)

if not matches:
    raise FileNotFoundError("Final ZIP was not found.")

src = max(matches, key=lambda p: p.stat().st_mtime)

print("Found:", src)
print(f"Size: {src.stat().st_size / 1024**2:.2f} MB")

# چک سالم بودن ZIP
with zipfile.ZipFile(src, "r") as z:
    bad = z.testzip()

if bad is not None:
    raise RuntimeError(f"ZIP is corrupted: {bad}")

print("ZIP integrity: OK")

# یک نام کوتاه برای جلوگیری از مشکل frontend
download_file = Path("/content/scDML_dataset4_results.zip")

if src.resolve() != download_file.resolve():
    shutil.copy2(src, download_file)

print("Ready:", download_file)
print(f"Download size: {download_file.stat().st_size / 1024**2:.2f} MB")

# دانلود مستقیم روی کامپیوتر
from google.colab import files

files.download(str(download_file))

## Runtime note

For the first run, temporarily use:

```python
METHOD_SEEDS = [42]
```

After the smoke run succeeds, restore:

```python
METHOD_SEEDS = [42, 123, 2026]
```

for the final benchmark.

This notebook is **statically validated**. It does not claim successful runtime
execution until the actual Dataset 4 + scDML Colab run completes.
